In [1]:
import numpy as np

def compute_mae(var, var_hat):
    return np.sum(np.abs(var - var_hat)) / var.shape[0]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])

In [2]:
import cvxpy as cp
import numpy as np

def robust_ar(y, d, p, delta, max_iter = 50):
    t = y.shape[0]
    vec = y[d :] # T-d
    mat = np.vstack([y[d-i-1 : t-i-1] for i in range(d)]).T # (T-d) x d
    theta = np.ones(t - d)
    for _ in range(max_iter):
        xt_w = mat.T * theta
        lhs = xt_w @ mat
        rhs = xt_w @ vec
        # w = np.linalg.pinv(lhs) @ rhs
        try:
            w = np.linalg.solve(lhs, rhs)
        except: #np.linalg.LinAlgError:
            w = np.linalg.lstsq(lhs, rhs, rcond = None)[0]
        epsilon = vec - mat @ w
        abs_eps = np.abs(epsilon)
        pos = np.where(abs_eps > delta)
        theta = np.ones(epsilon.shape)
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
    return w

def time_series_pred_multi(data, coef, horizon, step):
    t = data.shape[0]
    d = coef.shape[0]

    origins = np.arange(t - horizon, t) - step + 1             # (horizon,)
    assert origins.min() - d >= 0, "history too short"

    win = np.stack([data[o - d : o] for o in origins])         # horizon x d

    for _ in range(step):
        pred = win[:, ::-1] @ coef                             # (horizon,)
        win = np.hstack([win[:, 1:], pred[:, None]])

    return pred

(CVXPY) Oct 09 07:16:54 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Oct 09 07:16:54 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')


In [3]:
import pandas as pd

df = pd.read_csv('AI-small-data.csv', index_col = 0)

In [4]:
import numpy as np

x = df.loc['Machine_learning'].values[: 24 * 31]
num = 100
p = 1
delta = 20
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 55.8667
RMSE: 71.3757

MAE: 58.5964
RMSE: 73.929

MAE: 58.3369
RMSE: 73.6728

MAE: 58.3284
RMSE: 73.7492



In [5]:
import numpy as np

x = df.loc['Machine_learning'].values[: 24 * 31]
num = 100
p = 2/3
delta = 20
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 54.8231
RMSE: 70.2061

MAE: 59.7664
RMSE: 74.4078

MAE: 59.034
RMSE: 73.6831

MAE: 58.8027
RMSE: 73.3534



In [6]:
import numpy as np

x = df.loc['Machine_learning'].values[: 24 * 31]
num = 100
p = 1/2
delta = 20
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 56.6596
RMSE: 72.4581

MAE: 60.376
RMSE: 75.076

MAE: 59.9685
RMSE: 74.5488

MAE: 59.6112
RMSE: 74.0575



In [7]:
import numpy as np

x = df.loc['Machine_learning'].values[: 24 * 31]
num = 100
p = 1/3
delta = 20
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 56.6942
RMSE: 72.3355

MAE: 60.9585
RMSE: 75.8087

MAE: 60.2879
RMSE: 74.9284

MAE: 60.2049
RMSE: 74.9353



In [8]:
import numpy as np

x = df.loc['Machine_learning'].values[: 24 * 31]
num = 100
p = 0
delta = 20
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 59.4675
RMSE: 76.4749

MAE: 63.0248
RMSE: 78.6218

MAE: 61.0881
RMSE: 76.3965

MAE: 61.3185
RMSE: 76.6177



In [9]:
import numpy as np

x = df.loc['Bayesian_network'].values[: 24 * 31]
num = 100
p = 1
delta = 8
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 6.75662
RMSE: 8.56367

MAE: 6.31416
RMSE: 8.20736

MAE: 6.22717
RMSE: 8.10496

MAE: 6.24218
RMSE: 8.1264



In [10]:
import numpy as np

x = df.loc['Bayesian_network'].values[: 24 * 31]
num = 100
p = 2/3
delta = 8
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 7.23131
RMSE: 9.06848

MAE: 6.29859
RMSE: 8.19582

MAE: 6.22096
RMSE: 8.11816

MAE: 6.18015
RMSE: 8.07255



In [11]:
import numpy as np

x = df.loc['Bayesian_network'].values[: 24 * 31]
num = 100
p = 1/2
delta = 8
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 6.62786
RMSE: 8.44057

MAE: 6.30389
RMSE: 8.17694

MAE: 6.2425
RMSE: 8.16238

MAE: 6.11387
RMSE: 7.98779



In [12]:
import numpy as np

x = df.loc['Bayesian_network'].values[: 24 * 31]
num = 100
p = 1/3
delta = 8
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 7.13988
RMSE: 9.0082

MAE: 6.33826
RMSE: 8.21174

MAE: 6.25533
RMSE: 8.16184

MAE: 6.20413
RMSE: 8.10653



In [13]:
import numpy as np

x = df.loc['Bayesian_network'].values[: 24 * 31]
num = 100
p = 0
delta = 8
for degree in [1, 3, 7, 11]:
    mae = 0
    rmse = 0
    for i in range(num):
        np.random.seed(i)
        rng = np.random.default_rng()
        noise = rng.standard_t(degree, size=(24 * (31 - 7)))
        
        d = 168
        coef = robust_ar(x[: 24 * (31 - 14)] + noise[: 24 * (31 - 14)], d, p, delta)
        horizon = 168
        step = 12
        est = time_series_pred_multi(x, coef, horizon, step)
        
        mae += compute_mae((x[- horizon :]).reshape(-1), est.reshape(-1))
        rmse += compute_rmse((x[- horizon :]).reshape(-1), est.reshape(-1))
    print('MAE: {:.6}'.format(mae / num))
    print('RMSE: {:.6}'.format(rmse / num))
    print()

MAE: 6.9881
RMSE: 8.84211

MAE: 6.5089
RMSE: 8.42037

MAE: 6.42054
RMSE: 8.36517

MAE: 6.29607
RMSE: 8.23364

